In [ ]:
# ── Training Cell ───────────────────────────────────────────────────────────
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"]   = "1"

import numpy as np
import time
import sys
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

torch.backends.cudnn.benchmark     = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.enabled       = True

if torch.cuda.is_available():
    torch.cuda.empty_cache()

RANDLA_ROOT = Path(r"/mnt/e/Mona/Courses/AI/Project/Checkpoint_1/RandLA-Net-pytorch")
sys.path.insert(0, str(RANDLA_ROOT))
sys.path.insert(0, str(RANDLA_ROOT / "utils"))
os.chdir(RANDLA_ROOT)

from model import RandLANet
from utils.metrics import accuracy, intersection_over_union

try:
    from torch_points_kernels import knn as knn_fn
except ImportError:
    from torch_points import knn as knn_fn

# ── Parameters ────────────────────────────────────────────────────────────
DATA_DIR        = Path(r"/mnt/e/Mona/Courses/AI/Project/Checkpoint_1/RandLA-Net-pytorch/Dataset/Train")
LOGS_DIR        = RANDLA_ROOT / "runs" / "training"
CHECKPOINT_PATH = None        # set to a .pth path to resume, or None to start fresh

D_IN            = 5           # X Y Z Intensity PointSourceID
NUM_CLASSES     = 10
NUM_NEIGHBORS   = 16
DECIMATION      = 4
NUM_LAYERS      = 5
NUM_POINTS      = 20480       # must be divisible by 4^5 = 1024
EPOCHS          = 50
SAVE_FREQ       = 10
ADAM_LR         = 1e-2
SCHEDULER_GAMMA = 0.95
VAL_SPLIT       = 0.2         # 20% val, 80% train
BATCH_SIZE      = 1
NUM_WORKERS     = 0
SEP             = r"\s+"      # handles spaces or tabs
XYZ_COLS        = [0, 1, 2]   # X, Y, Z
FEAT_COLS       = [3, 4]      # Intensity, PointSourceID
LABEL_COL       = 5           # class label column
# ──────────────────────────────────────────────────────────────────────────

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device : {device}")
if device.type == "cuda":
    print(f"  GPU    : {torch.cuda.get_device_name(0)}")
    print(f"  Memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
LOGS_DIR.mkdir(parents=True, exist_ok=True)

# ── Move full inputs dict to device ───────────────────────────────────────
def inputs_to_device(inputs, device):
    return {
        'features':         inputs['features'].to(device),
        'coords':           [c.to(device) for c in inputs['coords']],
        'neighbor_indices': [n.to(device) for n in inputs['neighbor_indices']],
        'sub_idx':          [s.to(device) for s in inputs['sub_idx']],
        'interp_idx':       [i.to(device) for i in inputs['interp_idx']],
    }

# ── Precompute KNN inputs for all encoder layers ───────────────────────────
def build_inputs(pts, num_layers, num_neighbors, decimation):
    coords_list   = []
    neighbor_list = []
    sub_idx_list  = []
    interp_list   = []
    pc = pts[:, :3].copy()

    for i in range(num_layers):
        pc_tensor = torch.from_numpy(pc).unsqueeze(0)
        N_i       = pc.shape[0]

        neighbor_idx, _ = knn_fn(
            pc_tensor.contiguous(), pc_tensor.contiguous(), num_neighbors)

        N_sub  = N_i // decimation
        pool_i = neighbor_idx[:, :N_sub, :]
        pc_sub = pc[:N_sub, :]

        up_i, _ = knn_fn(
            torch.from_numpy(pc_sub).unsqueeze(0).contiguous(),
            pc_tensor.contiguous(), 1)

        coords_list.append(pc_tensor)
        neighbor_list.append(neighbor_idx.long())
        sub_idx_list.append(pool_i.long())
        interp_list.append(up_i.long())
        pc = pc_sub

    return {
        'features':         torch.from_numpy(pts).unsqueeze(0),
        'coords':           coords_list,
        'neighbor_indices': neighbor_list,
        'sub_idx':          sub_idx_list,
        'interp_idx':       interp_list,
    }

# ── Dataset ───────────────────────────────────────────────────────────────
class PointCloudDataset(Dataset):
    """
    Reads .txt tiles. Layout: X Y Z feat1 feat2 ... label
    Uses iloc for safe positional indexing regardless of pandas column naming.
    """
    def __init__(self, files, num_points, num_layers, num_neighbors,
                 decimation, xyz_cols, feat_cols, label_col, sep):
        self.files         = files
        self.num_points    = num_points
        self.num_layers    = num_layers
        self.num_neighbors = num_neighbors
        self.decimation    = decimation
        self.xyz_cols      = xyz_cols
        self.feat_cols     = feat_cols
        self.label_col     = label_col
        self.sep           = sep

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        import pandas as pd
        df = pd.read_csv(self.files[idx], sep=self.sep,
                         header=None, engine="python")

        # iloc = positional indexing, safe regardless of column names
        xyz    = df.iloc[:, self.xyz_cols].values.astype(np.float32)
        feat   = df.iloc[:, self.feat_cols].values.astype(np.float32)
        labels = df.iloc[:, self.label_col].values.astype(np.int64)

        # Normalize features to [0, 1]
        f_min = feat.min(axis=0, keepdims=True)
        f_max = feat.max(axis=0, keepdims=True)
        feat  = (feat - f_min) / (f_max - f_min + 1e-8)

        pts = np.hstack([xyz, feat]).astype(np.float32)   # (N, D_IN)

        # Sample / pad to fixed NUM_POINTS
        N = len(pts)
        if N >= self.num_points:
            idx_s = np.random.choice(N, self.num_points, replace=False)
        else:
            idx_s = np.concatenate([
                np.arange(N),
                np.random.choice(N, self.num_points - N, replace=True)
            ])
        pts    = pts[idx_s]
        labels = np.clip(labels[idx_s], 0, NUM_CLASSES - 1)

        inputs = build_inputs(pts, self.num_layers,
                              self.num_neighbors, self.decimation)
        return inputs, torch.from_numpy(labels)

# ── Collate fn ────────────────────────────────────────────────────────────
def collate_fn(batch):
    inputs_list, labels_list = zip(*batch)
    labels = torch.stack(labels_list, dim=0)   # (B, N)

    def stack_layers(tensor_lists):
        n_layers = len(tensor_lists[0])
        return [
            torch.cat([tensor_lists[b][i]
                       for b in range(len(tensor_lists))], dim=0)
            for i in range(n_layers)
        ]

    inputs = {
        'features':         torch.cat([inp['features']         for inp in inputs_list], dim=0),
        'coords':           stack_layers([inp['coords']           for inp in inputs_list]),
        'neighbor_indices': stack_layers([inp['neighbor_indices'] for inp in inputs_list]),
        'sub_idx':          stack_layers([inp['sub_idx']          for inp in inputs_list]),
        'interp_idx':       stack_layers([inp['interp_idx']       for inp in inputs_list]),
    }
    return inputs, labels

# ── Train / val file split ────────────────────────────────────────────────
all_files = sorted(DATA_DIR.glob("*.txt"))
print(f"\nTotal tile files found : {len(all_files)}")
if len(all_files) == 0:
    raise FileNotFoundError(f"No .txt files found in '{DATA_DIR}'")

n_val       = max(1, int(len(all_files) * VAL_SPLIT))
n_train     = len(all_files) - n_val
perm        = torch.randperm(len(all_files)).tolist()
train_files = [all_files[i] for i in perm[:n_train]]
val_files   = [all_files[i] for i in perm[n_train:]]
print(f"Train : {len(train_files)} files ({100*(1-VAL_SPLIT):.0f}%)")
print(f"Val   : {len(val_files)}   files ({100*VAL_SPLIT:.0f}%)")

dataset_kwargs = dict(
    num_points    = NUM_POINTS,
    num_layers    = NUM_LAYERS,
    num_neighbors = NUM_NEIGHBORS,
    decimation    = DECIMATION,
    xyz_cols      = XYZ_COLS,
    feat_cols     = FEAT_COLS,
    label_col     = LABEL_COL,
    sep           = SEP,
)
train_dataset = PointCloudDataset(train_files, **dataset_kwargs)
val_dataset   = PointCloudDataset(val_files,   **dataset_kwargs)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                            shuffle=True,  num_workers=NUM_WORKERS,
                            collate_fn=collate_fn)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=NUM_WORKERS,
                            collate_fn=collate_fn)
print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")

# ── Model ─────────────────────────────────────────────────────────────────
print("\nBuilding model ...")
model     = RandLANet(D_IN, NUM_CLASSES, NUM_NEIGHBORS, DECIMATION, device)
optimizer = torch.optim.Adam(model.parameters(), lr=ADAM_LR)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, SCHEDULER_GAMMA)
# NLLLoss + log_softmax = correct way when model outputs raw scores
criterion = nn.NLLLoss()

first_epoch = 1
if CHECKPOINT_PATH is not None:
    print(f"Resuming from '{CHECKPOINT_PATH}' ...")
    ckpt        = torch.load(CHECKPOINT_PATH, map_location=device)
    first_epoch = ckpt['epoch'] + 1
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    print(f"  Resuming from epoch {first_epoch}")

# ── Evaluate ──────────────────────────────────────────────────────────────
def evaluate(model, loader, criterion, device):
    model.eval()
    losses, accuracies, ious = [], [], []
    with torch.no_grad():
        for inputs, labels in tqdm(loader, desc='Validation', leave=False):
            # move everything to device
            labels = labels.to(device)
            inputs = inputs_to_device(inputs, device)
            try:
                scores    = model(inputs)                        # (B, N, C)
                scores_t  = scores.permute(0, 2, 1)             # (B, C, N)
                log_probs = torch.log_softmax(scores_t, dim=1)  # (B, C, N)
                loss      = criterion(log_probs, labels)
                losses.append(loss.cpu().item())
                for b in range(scores_t.size(0)):
                    # metrics expect (C, N) and (N,)
                    accuracies.append(accuracy(scores_t[b], labels[b]))
                    ious.append(intersection_over_union(scores_t[b], labels[b]))
            except RuntimeError as e:
                print(f"\n  [Val WARNING] Skipping batch: {e}")
                torch.cuda.empty_cache()
                continue
    return (np.mean(losses)        if losses     else 0.0,
            np.nanmean(accuracies) if accuracies else 0.0,
            np.nanmean(ious)       if ious        else 0.0)

# ── Training loop ─────────────────────────────────────────────────────────
print("\nStarting training ...")
with SummaryWriter(str(LOGS_DIR)) as writer:
    for epoch in range(first_epoch, EPOCHS + 1):
        print(f"\n=== EPOCH {epoch}/{EPOCHS} ===")
        t0 = time.time()
        model.train()
        losses, accuracies, ious = [], [], []

        for inputs, labels in tqdm(train_loader, desc='Training', leave=False):
            # move everything to device
            labels = labels.to(device)
            inputs = inputs_to_device(inputs, device)
            optimizer.zero_grad()

            try:
                scores    = model(inputs)                        # (B, N, C)
                scores_t  = scores.permute(0, 2, 1)             # (B, C, N)
                log_probs = torch.log_softmax(scores_t, dim=1)  # (B, C, N)
                loss      = criterion(log_probs, labels)
                loss.backward()
                optimizer.step()

                losses.append(loss.cpu().item())
                for b in range(scores_t.size(0)):
                    # metrics expect (C, N) and (N,)
                    accuracies.append(accuracy(scores_t[b], labels[b]))
                    ious.append(intersection_over_union(scores_t[b], labels[b]))

            except RuntimeError as e:
                print(f"\n  [Train WARNING] Skipping batch: {e}")
                torch.cuda.empty_cache()
                continue

        scheduler.step()

        if not losses:
            print("  No batches completed — skipping metrics.")
            continue

        val_loss, val_acc, val_iou = evaluate(model, val_loader, criterion, device)

        t1      = time.time()
        d       = t1 - t0
        tr_loss = np.mean(losses)
        # accuracies / ious are lists of lists — flatten before nanmean
        tr_acc  = np.nanmean([a for sub in accuracies for a in sub])
        tr_iou  = np.nanmean([i for sub in ious       for i in sub])

        print(f"  Train — Loss: {tr_loss:.5f}  Acc: {tr_acc:.4f}  mIoU: {tr_iou:.4f}")
        print(f"  Val   — Loss: {val_loss:.5f}  Acc: {val_acc:.4f}  mIoU: {val_iou:.4f}")
        print(f"  Time  — {'%.0f s' % d if d < 60 else '%.0f min %02.0f s' % divmod(d, 60)}")

        if device.type == "cuda":
            mem = torch.cuda.memory_reserved(0) / 1e9
            print(f"  GPU mem : {mem:.2f} GB")
            torch.cuda.empty_cache()

        writer.add_scalars('Loss',     {'Train': tr_loss, 'Val': val_loss}, epoch)
        writer.add_scalars('Accuracy', {'Train': tr_acc,  'Val': val_acc},  epoch)
        writer.add_scalars('mIoU',     {'Train': tr_iou,  'Val': val_iou},  epoch)

        if epoch % SAVE_FREQ == 0:
            ckpt_path = LOGS_DIR / f"checkpoint_{epoch:03d}.pth"
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
            }, ckpt_path)
            print(f"  Checkpoint → '{ckpt_path}'")

print("\nTraining complete.")

Using device : cuda:0
  GPU    : NVIDIA RTX A4000
  Memory : 17.2 GB

Total tile files found : 2
Train : 1 files (80%)
Val   : 1   files (20%)
Train batches : 1
Val batches   : 1

Building model ...

Starting training ...

=== EPOCH 1/50 ===


  Train — Loss: 2.39902  Acc: 0.0937  mIoU: 0.0312
  Val   — Loss: 217.09053  Acc: 0.3187  mIoU: 0.1483
  Time  — 27 s
  GPU mem : 0.62 GB

=== EPOCH 2/50 ===


  Train — Loss: 1.98133  Acc: 0.2153  mIoU: 0.0858
  Val   — Loss: 7.63837  Acc: 0.3178  mIoU: 0.1472
  Time  — 25 s
  GPU mem : 0.66 GB

=== EPOCH 3/50 ===


  Train — Loss: 1.74490  Acc: 0.2774  mIoU: 0.1196
  Val   — Loss: 1.16725  Acc: 0.4404  mIoU: 0.2290
  Time  — 27 s
  GPU mem : 0.66 GB

=== EPOCH 4/50 ===


  Train — Loss: 1.51895  Acc: 0.3160  mIoU: 0.1326
  Val   — Loss: 1.26008  Acc: 0.4789  mIoU: 0.2099
  Time  — 26 s
  GPU mem : 0.66 GB

=== EPOCH 5/50 ===


KeyboardInterrupt: 